# Cài đặt thư viện

In [ ]:
!pip install -q datasets==3.2.0
!pip install --upgrade huggingface-hub evaluate

# Load dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("thainq107/abte-restaurants")

ds

# Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

# Tiền xử lý dữ liệu

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = []
    labels = []

    for tokens, tags in zip(examples['Tokens'], examples['Tags']):
        bert_tokens = []
        bert_tags = []
        for i in range(len(tokens)):
            t = tokenizer.tokenize(tokens[i])
            bert_tokens += t
            bert_tags += [int(tags[i])] * len(t)
        
        bert_ids = tokenizer.convert_tokens_to_ids(bert_tokens)
        tokenized_inputs.append(bert_ids)
        labels.append(bert_tags)
    
    return {"input_ids": tokenized_inputs, "labels": labels}

In [ ]:
preprocessed_ds = ds.map(tokenize_and_align_labels, batched=True)
preprocessed_ds

# Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "O",
    1: "B-Term",
    2: "I-Term"
}
label2id = {
    "O": 0,
    "B-Term": 1,
    "I-Term": 2
}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
model

# Huấn luyện mô hình

In [ ]:
!pip install -q seqeval==1.2.2

In [ ]:
import numpy as np
from seqeval.metrics import accuracy_score, f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100] # -100: [CLS], [SEP]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = f1_score(true_predictions, true_labels)
    return {"F1-score": results}

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./abte-restaurants-distilbert-base-uncased",
    logging_dir="logs",
    learning_rate=2e-5,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    num_train_epochs=30,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="F1-score",
    # report_to="wandb",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_ds["train"],
    eval_dataset=preprocessed_ds["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

# Kiểm thử mô hình

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

model = AutoModelForTokenClassification.from_pretrained("./abte-restaurants-distilbert-base-uncased/checkpoint-450")

id2label = {0: "O", 1: "B-Term", 2: "I-Term"}

ate_pipe = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

def extract_aspects(text):
    predictions = ate_pipe(text)
    aspects = []
    for entity in predictions:
        if entity['entity_group'] == 'Term': 
            aspects.append(entity['word'])
    return aspects

text = "The price was too high, but the cab was amazing."
aspects = extract_aspects(text)
print(f"Extracted aspects: {aspects}")

In [ ]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from huggingface_hub import login

# 1. Dang nhap Hugging Face
login()

# 2. Cau hinh
checkpoint_path = "/content/abte-restaurants-distilbert-base-uncased/checkpoint-450"
tokenizer_name = "distilbert/distilbert-base-uncased"
repo_id = "GinNoV111/abte-restaurant-model"   

# 3. Load model va tokenizer
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

id2label = {
    0: "O",
    1: "B-Term",
    2: "I-Term"
}
label2id = {
    "O": 0,
    "B-Term": 1,
    "I-Term": 2
}

model.config.id2label = id2label
model.config.label2id = label2id

# 4. Push len Hugging Face Hub
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Da push model len: https://huggingface.co/{repo_id}")